# 05 Feature Modelling

This notebook tests whether lag features, rolling demand features, calendar features and selected exogenous variables can improve over the year-over-year seasonal naive benchmark, especially on high-demand days. It does not add Prophet, simulation or deep learning.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src import feature_models
from src.utils import FIGURES_DIR, TABLES_DIR

TARGET = "nd_mean"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

## Load Processed Daily Dataset

In [ ]:
daily = feature_models.load_daily_dataset()
print("Shape:", daily.shape)
display(daily.head())

## Feature Engineering

The modelling pipeline creates target lags at 1, 2, 3, 7, 14, 28 and 365 days. Rolling demand features are shifted by one day before calculation, so they do not include the forecast day's actual target. Demand regimes are defined from training-period quantiles only.

In [ ]:
features = feature_models.create_calendar_features(
    feature_models.create_rolling_features(
        feature_models.create_lag_features(daily, target=TARGET),
        target=TARGET,
    )
)
train_mask = features["date"] < pd.Timestamp(TEST_START)
features, thresholds = feature_models.create_demand_regime_labels(features, TARGET, train_mask)
feature_columns = feature_models.select_feature_columns(features, TARGET)
print("Demand regime thresholds:", thresholds)
print("Feature count:", len(feature_columns))
print(feature_columns)

## Run Feature Models

In [ ]:
comparison, regime_comparison, forecasts = feature_models.run_feature_modelling(
    target=TARGET,
    test_start=TEST_START,
    test_end=TEST_END,
)
display(comparison)
display(regime_comparison)
display(forecasts.head())

## Feature Importance

In [ ]:
importance = pd.read_csv(TABLES_DIR / "feature_model_importance.csv")
ridge_coefficients = pd.read_csv(TABLES_DIR / "ridge_feature_coefficients.csv")
display(importance.groupby("model").head(15))
display(ridge_coefficients.head(20))

## Figures

In [ ]:
for figure_path in [
    FIGURES_DIR / "modelling" / "actual_vs_feature_model_forecasts.png",
    FIGURES_DIR / "modelling" / "feature_model_mape_comparison.png",
    FIGURES_DIR / "modelling" / "feature_model_regime_mape_comparison.png",
    FIGURES_DIR / "modelling" / "feature_importance_top20.png",
]:
    print(figure_path)
    if figure_path.exists():
        image = plt.imread(figure_path)
        plt.figure(figsize=(14, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

## Interpretation

The year-over-year seasonal naive benchmark remains the model to beat. A feature model should only be described as better if the saved metrics show lower overall error on the same 2025 test period. High-demand performance should be reviewed separately using `feature_model_regime_comparison.csv`; this matters because later scenario work will focus on capacity-pressure days.

If feature models do not beat seasonal naive overall, or do not improve high-demand MAPE, the report should say so plainly and recommend further feature review rather than moving directly to Prophet or simulation.